# 19 — Financial Model Builder

**What it does:** Extract financial assumptions from a business brief and compute a validated 3-year P&L.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/19-financial-modeller/financial_modeller_workbook.ipynb)

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 1. How it works

Two steps:
1. **LLM reads** the brief and fills in a `FinancialAssumptions` schema (rates, amounts, growth)
2. **Python computes** the 3-year model deterministically from those typed parameters

The AI handles ambiguous language; Python handles the arithmetic.

In [ ]:
from pydantic import BaseModel, Field

class FinancialAssumptions(BaseModel):
    revenue_y1: float = Field(description='Year 1 revenue in USD.')
    revenue_growth_rate: float = Field(description='Annual revenue growth rate (0.15 = 15%).')
    cogs_pct: float = Field(description='COGS as fraction of revenue (0.40 = 40%).')
    opex_y1: float = Field(description='Year 1 opex in USD.')
    opex_growth_rate: float = Field(description='Annual opex growth rate.')
    tax_rate: float = Field(description='Tax rate as decimal.')
    capex_y1: float = Field(description='Year 1 capex in USD.')
    depreciation_y1: float = Field(description='Year 1 depreciation in USD.')
    debt_service_annual: float = Field(description='Annual debt service. 0.0 if no debt.')

class YearlyProjection(BaseModel):
    year: int
    revenue: float
    cogs: float
    gross_profit: float
    opex: float
    ebitda: float
    tax: float
    net_income: float
    fcf: float

class FinancialModel(BaseModel):
    assumptions: FinancialAssumptions
    projections: list[YearlyProjection]
    dscr: float
    is_viable: bool
    viability_notes: str

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

def _extract(brief: str) -> FinancialAssumptions:
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    chain = SystemMessage('Extract financial assumptions. Rates as decimals. Amounts in USD. Use 0.0 for missing debt.') | llm.with_structured_output(FinancialAssumptions)
    return chain.invoke(brief)

def _project(a: FinancialAssumptions) -> FinancialModel:
    projections, capex, dep = [], a.capex_y1, a.depreciation_y1
    for yr in range(1, 4):
        rev = a.revenue_y1 * (1 + a.revenue_growth_rate) ** (yr - 1)
        cogs = rev * a.cogs_pct
        opex = a.opex_y1 * (1 + a.opex_growth_rate) ** (yr - 1)
        ebitda = rev - cogs - opex
        tax = max(0.0, ebitda * a.tax_rate)
        ni = ebitda - tax
        fcf = ebitda - tax - capex + dep
        projections.append(YearlyProjection(year=yr, revenue=round(rev,2), cogs=round(cogs,2), gross_profit=round(rev-cogs,2), opex=round(opex,2), ebitda=round(ebitda,2), tax=round(tax,2), net_income=round(ni,2), fcf=round(fcf,2)))
        capex *= 0.6; dep *= 1.1
    avg_ebitda = sum(p.ebitda for p in projections) / 3
    dscr = round(avg_ebitda / a.debt_service_annual, 2) if a.debt_service_annual > 0 else 0.0
    y2, y3 = projections[1], projections[2]
    is_viable = y3.net_income > 0 and y2.fcf > 0
    concerns = ([f'net income negative Y3'] if y3.net_income <= 0 else []) + ([f'FCF negative Y2'] if y2.fcf <= 0 else [])
    return FinancialModel(assumptions=a, projections=projections, dscr=dscr, is_viable=is_viable, viability_notes='Viable' if is_viable else f'Concerns: {chr(44).join(concerns)}')

def run(brief: str) -> FinancialModel:
    return _project(_extract(brief))

## 2. Run — two business types

In [ ]:
saas = run('B2B SaaS. Year-1 ARR $1.2M, growing 45% annually. COGS 25% of revenue. Opex $800K year 1, growing 20%. Capex $150K, depreciation $40K/year. No debt. Tax 25%.')
print(f'SaaS viability: {saas.viability_notes}')
for p in saas.projections:
    print(f'  Y{p.year}: revenue=${p.revenue:,.0f}  EBITDA=${p.ebitda:,.0f}  FCF=${p.fcf:,.0f}')

In [ ]:
mfg = run('Industrial manufacturer. Year-1 revenue $4.5M, 12% growth. COGS 55%. Opex $1.1M growing 8%. Capex $600K, depreciation $80K. Debt service $280K/year. Tax 28%.')
print(f'Manufacturing viability: {mfg.viability_notes}')
if mfg.dscr > 0:
    print(f'DSCR: {mfg.dscr:.2f}x')

## 3. Exercise

Add a `break_even_year` field to `FinancialModel` that reports which year FCF first turns positive (or `None` if it never does across the 3-year horizon).

In [ ]:
# Your code here

### Answer Key

In [ ]:
from typing import Optional

class FinancialModelV2(FinancialModel):
    break_even_year: Optional[int] = Field(default=None, description='First year FCF is positive, or None.')

def run_v2(brief: str) -> FinancialModelV2:
    base = run(brief)
    bey = next((p.year for p in base.projections if p.fcf > 0), None)
    return FinancialModelV2(**base.model_dump(), break_even_year=bey)

r = run_v2('SaaS. ARR $1.2M, 45% growth, COGS 25%, opex $800K, capex $150K, dep $40K, no debt, tax 25%.')
print(f'Break-even FCF year: {r.break_even_year}')